# Phase 3 — Feature Engineering

## Life Claims Intelligence

### Objective

Create and validate business-critical derived metrics required for the
Life Claims Intelligence analysis.

The feature engineering process focuses on claim settlement efficiency,
claims outcomes, pending claims and ageing, and unclaimed funds.

### Final Features

#### Settlement Efficiency
1. Count_CSR
2. Value_CSR
3. Count_Value_Gap

#### Claims Outcomes
4. Repudiation_Rate

#### Pending Claims & Ageing
5. Pending_Rate
6. Long_Pending_Share

#### Unclaimed Funds
7. Unclaimed_Ratio

### Feature Selection Principle

Only features with clear business relevance and sufficient data coverage
are retained in the final analytical dataset.

Conceptually relevant features with insufficient data coverage are
excluded rather than artificially imputed.

### Data Integrity Principle

Raw data remains unchanged.

Feature engineering is performed only on the cleaned dataset generated
in Phase 2.

All engineered data is saved as a separate downstream dataset.

----

### Importing

In [50]:
import pandas as pd
import numpy as np

----

### Load CSV

In [51]:
df = pd.read_csv(r"D:\My Projects\Insurance_Claims_Intelligence\data\clean\Life_Claims_Cleaned.csv")
df

,FY,Insurer,pending_start_count,pending_start_amount,intimated_count,intimated_amount,total_claims_count,total_claims_amount,paid_count,paid_amount,...,rejected_amount,unclaimed_count,unclaimed_amount,pending_end_count,pending_end_amount,pending_lt_3m,pending_3_to_6m,pending_6m_to_1y,pending_gt_1y,Low_Volume_Flag
0,2020-21,LIC,5875,349.690000,941101,18755.650000,946976,19105.340000,933889,18295.580000,...,3.920000,1897,236.490000,1725,2.924200e+02,792.0,933.0,0.0,0.0,False
1,2020-21,Aditya Birla Sun Life,19,3.802883,6455,468.846357,6474,472.649240,6347,440.264288,...,0.000000,0,0.000000,11,3.637908e+00,10.0,1.0,0.0,0.0,False
2,2020-21,Bandhan,0,0.000000,401,107.440000,401,107.440000,398,105.980000,...,0.000000,0,0.000000,0,0.000000e+00,0.0,0.0,0.0,0.0,True
3,2020-21,Ageas Federal,5,1.255000,1800,87.051683,1805,88.306683,1716,73.848647,...,0.048501,0,0.000000,50,6.698502e+00,50.0,0.0,0.0,0.0,False
4,2020-21,Aviva,5,0.778359,1050,116.371251,1055,117.149610,1034,111.572118,...,0.000000,0,0.000000,0,1.065814e-14,0.0,0.0,0.0,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,2024-25,Sahara,0,0.000000,0,0.000000,0,0.000000,0,0.000000,...,0.000000,0,0.000000,0,0.000000e+00,0.0,0.0,0.0,0.0,True
118,2024-25,SBI Life,118,23.580493,44831,2598.751947,44949,2622.332439,44205,2497.939508,...,0.000000,55,7.610102,165,4.027212e+01,106.0,23.0,34.0,2.0,False
119,2024-25,Shriram,6,0.294771,4846,180.195625,4852,180.490395,4770,147.476986,...,10.150834,0,0.000000,7,1.977760e-01,7.0,0.0,0.0,0.0,False
120,2024-25,Star Union,3,0.805000,2410,144.384758,2413,145.189758,2385,140.738808,...,0.000000,0,0.000000,2,5.757500e-01,2.0,0.0,0.0,0.0,False


### Validate

In [52]:
# shape 

df.shape

(122, 23)

In [53]:
df.columns.tolist()

['FY',
 'Insurer',
 'pending_start_count',
 'pending_start_amount',
 'intimated_count',
 'intimated_amount',
 'total_claims_count',
 'total_claims_amount',
 'paid_count',
 'paid_amount',
 'repudiated_count',
 'repudiated_amount',
 'rejected_count',
 'rejected_amount',
 'unclaimed_count',
 'unclaimed_amount',
 'pending_end_count',
 'pending_end_amount',
 'pending_lt_3m',
 'pending_3_to_6m',
 'pending_6m_to_1y',
 'pending_gt_1y',
 'Low_Volume_Flag']

---

## 1. Count Claim Settlement Ratio (Count_CSR)

### Why are we creating this feature?

Count_CSR measures the percentage of total claims that were paid by an insurer.

It helps us evaluate claim settlement efficiency based on the **number of claims handled**, rather than the monetary value of those claims.

### Formula

Count_CSR = (Paid Claims Count / Total Claims Count) × 100

### Business Interpretation

A higher Count_CSR indicates that a larger proportion of reported claims were paid.

This metric will help us:
- Compare claim settlement performance across insurers.
- Track changes in settlement efficiency over financial years.
- Identify insurers with relatively lower claim settlement performance.

### Data Handling

If `total_claims_count` is zero, it is replaced with `NaN` before division to avoid division-by-zero errors and misleading infinite values.

---

In [54]:
df['Count_CSR'] = (
    df['paid_count'] /
    df['total_claims_count'].replace(0, np.nan)
) * 100

In [55]:
df[['Insurer', 'FY', 'total_claims_count', 'paid_count', 'Count_CSR']].head(10)

,Insurer,FY,total_claims_count,paid_count,Count_CSR
0,LIC,2020-21,946976,933889,98.618022
1,Aditya Birla Sun Life,2020-21,6474,6347,98.038307
2,Bandhan,2020-21,401,398,99.251870
3,Ageas Federal,2020-21,1805,1716,95.069252
4,Aviva,2020-21,1055,1034,98.009479
5,Bajaj Allianz,2020-21,14333,14115,98.479034
6,Bharti Axa,2020-21,1893,1875,99.049128
7,Canara HSBC,2020-21,1899,1844,97.103739
8,Edelweiss Life,2020-21,502,487,97.011952
9,Exide Life,2020-21,5052,4978,98.535234


----

## 2. Value Claim Settlement Ratio (Value_CSR)

### Why are we creating this feature?

Count_CSR tells us how many claims were paid, but it does not tell us how much claim value was actually paid.

Value_CSR measures the percentage of total claim amount that was paid by the insurer.

This gives us a **monetary perspective** of claim settlement performance.

### Formula

Value_CSR = (Paid Claim Amount / Total Claim Amount) × 100

### Business Interpretation

A higher Value_CSR indicates that a larger proportion of the total claim value was paid.

Comparing Value_CSR with Count_CSR will help us identify whether settlement performance looks different when measured by **claim volume versus claim value**.

### Data Handling

If `total_claims_amount` is zero, it is replaced with `NaN` before division to avoid division-by-zero errors.

----

In [56]:
df['Value_CSR'] = (
    df['paid_amount'] /
    df['total_claims_amount'].replace(0, np.nan)
) * 100

In [57]:
df[['Insurer', 'FY', 'total_claims_amount', 'paid_amount', 'Value_CSR']].head(10)

,Insurer,FY,total_claims_amount,paid_amount,Value_CSR
0,LIC,2020-21,19105.340000,18295.580000,95.761604
1,Aditya Birla Sun Life,2020-21,472.649240,440.264288,93.148206
2,Bandhan,2020-21,107.440000,105.980000,98.641102
3,Ageas Federal,2020-21,88.306683,73.848647,83.627473
4,Aviva,2020-21,117.149610,111.572118,95.239000
5,Bajaj Allianz,2020-21,446.536349,410.676805,91.969401
6,Bharti Axa,2020-21,107.598328,106.035205,98.547260
7,Canara HSBC,2020-21,168.537924,156.075726,92.605701
8,Edelweiss Life,2020-21,52.089138,45.828280,87.980492
9,Exide Life,2020-21,182.704017,170.430141,93.282098


---

## 3. Count-Value Settlement Gap (Count_Value_Gap)

### Why are we creating this feature?

Count_CSR measures settlement performance based on the number of claims,
while Value_CSR measures settlement performance based on the monetary value
of claims.

An insurer may have a high settlement rate by claim count but a lower
settlement rate by claim value. This can indicate that the settlement
performance differs depending on the size/value of claims.

Count_Value_Gap captures this difference in a single metric.

### Formula

Count_Value_Gap = Count_CSR - Value_CSR

### Business Interpretation

- Positive gap → Count_CSR is higher than Value_CSR.
- Negative gap → Value_CSR is higher than Count_CSR.
- Gap close to zero → Count-based and value-based settlement performance
  are broadly similar.

A large positive gap can indicate that smaller-value claims are being
settled at a higher rate than larger-value claims.

### Data Handling

The feature uses the already calculated Count_CSR and Value_CSR values,
so no additional division is required.

----

In [58]:
df['Count_Value_Gap'] = df['Count_CSR'] - df['Value_CSR']

In [59]:
df[['Insurer', 'FY', 'Count_CSR', 'Value_CSR', 'Count_Value_Gap']].head(10)

,Insurer,FY,Count_CSR,Value_CSR,Count_Value_Gap
0,LIC,2020-21,98.618022,95.761604,2.856418
1,Aditya Birla Sun Life,2020-21,98.038307,93.148206,4.890101
2,Bandhan,2020-21,99.251870,98.641102,0.610768
3,Ageas Federal,2020-21,95.069252,83.627473,11.441779
4,Aviva,2020-21,98.009479,95.239000,2.770478
5,Bajaj Allianz,2020-21,98.479034,91.969401,6.509633
6,Bharti Axa,2020-21,99.049128,98.547260,0.501868
7,Canara HSBC,2020-21,97.103739,92.605701,4.498038
8,Edelweiss Life,2020-21,97.011952,87.980492,9.031461
9,Exide Life,2020-21,98.535234,93.282098,5.253135


----

## 4. Claim Value Disparity (Claim_Value_Disparity)

### Why are we creating this feature?

Count_CSR and Value_CSR tell us about overall settlement performance,
but they do not show whether rejected claims tend to have a different
average value compared with paid claims.

Claim_Value_Disparity compares the **average value of rejected claims**
with the **average value of paid claims**.

### Formula

Average Rejected Claim Value = Rejected Amount / Rejected Claims Count

Average Paid Claim Value = Paid Amount / Paid Claims Count

Claim_Value_Disparity =
Average Rejected Claim Value / Average Paid Claim Value

### Business Interpretation

- Value > 1 → rejected claims have a higher average value than paid claims.
- Value < 1 → rejected claims have a lower average value than paid claims.
- Value ≈ 1 → rejected and paid claims have similar average values.

This helps identify whether rejection patterns are disproportionately
associated with higher- or lower-value claims.

### Data Handling

If `rejected_count` or `paid_count` is zero, the corresponding denominator
is replaced with `NaN` to avoid division-by-zero errors and misleading
infinite values.

----

In [60]:
df['Claim_Value_Disparity'] = (
    (df['rejected_amount'] / df['rejected_count'].replace(0, np.nan)) /
    (df['paid_amount'] / df['paid_count'].replace(0, np.nan))
)

In [61]:
df[
    ['Insurer', 'FY',
     'rejected_count', 'rejected_amount',
     'paid_count', 'paid_amount',
     'Claim_Value_Disparity']
].head(10)

,Insurer,FY,rejected_count,rejected_amount,paid_count,paid_amount,Claim_Value_Disparity
0,LIC,2020-21,2934,3.920000,933889,18295.580000,0.068199
1,Aditya Birla Sun Life,2020-21,0,0.000000,6347,440.264288,NaN
2,Bandhan,2020-21,0,0.000000,398,105.980000,NaN
3,Ageas Federal,2020-21,1,0.048501,1716,73.848647,1.126992
4,Aviva,2020-21,0,0.000000,1034,111.572118,NaN
5,Bajaj Allianz,2020-21,0,0.000000,14115,410.676805,NaN
6,Bharti Axa,2020-21,0,0.000000,1875,106.035205,NaN
7,Canara HSBC,2020-21,0,0.000000,1844,156.075726,NaN
8,Edelweiss Life,2020-21,0,0.000000,487,45.828280,NaN
9,Exide Life,2020-21,0,0.000000,4978,170.430141,NaN


### Validation

A disparity value cannot be calculated when an insurer has zero rejected
claims or zero paid claims. These cases are retained as `NaN` rather than
being replaced with zero, because the metric is mathematically undefined
in such cases.

In [62]:
df['Claim_Value_Disparity'].isna().sum()

np.int64(103)

In [63]:
np.isinf(df['Claim_Value_Disparity']).sum()

np.int64(0)

---

## 5. Unclaimed Amount Ratio (Unclaimed_Ratio)

### Why are we creating this feature?

Claim settlement performance does not tell the complete policyholder
outcome. An amount may remain unclaimed even after claims activity has
taken place.

Unclaimed_Ratio measures the unclaimed claim amount relative to the
total claim amount reported for an insurer-year.

This helps us assess the scale of unclaimed funds in relation to the
overall claims value.

### Formula

Unclaimed_Ratio = (Unclaimed Amount / Total Claim Amount) × 100

### Business Interpretation

- Lower ratio → unclaimed amount is relatively small compared with total
  claim value.
- Higher ratio → a relatively larger portion of claim value remains
  unclaimed.

This metric can help us investigate whether the scale of unclaimed
amounts has changed across insurers and financial years.

### Data Handling

If `total_claims_amount` is zero, it is replaced with `NaN` before
division to avoid division-by-zero errors and misleading infinite values.

----

In [64]:
df['Unclaimed_Ratio'] = (
    df['unclaimed_amount'] /
    df['total_claims_amount'].replace(0, np.nan)
) * 100

In [65]:
df[
    ['Insurer', 'FY',
     'total_claims_amount',
     'unclaimed_amount',
     'Unclaimed_Ratio']
].head(30)

,Insurer,FY,total_claims_amount,unclaimed_amount,Unclaimed_Ratio
0,LIC,2020-21,19105.340000,236.490000,1.237821
1,Aditya Birla Sun Life,2020-21,472.649240,0.000000,0.000000
2,Bandhan,2020-21,107.440000,0.000000,0.000000
3,Ageas Federal,2020-21,88.306683,0.000000,0.000000
4,Aviva,2020-21,117.149610,0.000000,0.000000
5,Bajaj Allianz,2020-21,446.536349,0.000000,0.000000
6,Bharti Axa,2020-21,107.598328,0.000000,0.000000
7,Canara HSBC,2020-21,168.537924,0.000000,0.000000
8,Edelweiss Life,2020-21,52.089138,0.000000,0.000000
9,Exide Life,2020-21,182.704017,0.000000,0.000000


In [66]:
df['Unclaimed_Ratio'].isna().sum()

np.int64(6)

In [67]:
np.isinf(df['Unclaimed_Ratio']).sum()

np.int64(0)

---

## 6. Pending Claims Rate (Pending_Rate)

### Why are we creating this feature?

Claim Settlement Ratio measures the proportion of claims that were paid,
but it does not show how many claims remain unresolved at the end of the
financial year.

Pending_Rate measures the proportion of total claims that remain pending
at the end of the year.

This provides an additional view of an insurer's claims handling and
unresolved claim burden.

### Formula

Pending_Rate = (Pending Claims at End / Total Claims Count) × 100

### Business Interpretation

- Lower Pending_Rate → relatively fewer claims remain unresolved.
- Higher Pending_Rate → relatively larger unresolved claim burden.
  
This metric can be used alongside Count_CSR to identify insurers that may
have a high settlement rate but still carry a relatively high pending
claim burden.

### Data Handling

If `total_claims_count` is zero, it is replaced with `NaN` before division
to avoid division-by-zero errors and misleading infinite values.

----

In [68]:
df['Pending_Rate'] = (
    df['pending_end_count'] /
    df['total_claims_count'].replace(0, np.nan)
) * 100

In [69]:
df[
    ['Insurer', 'FY',
     'total_claims_count',
     'pending_end_count',
     'Pending_Rate']
].head(10)

,Insurer,FY,total_claims_count,pending_end_count,Pending_Rate
0,LIC,2020-21,946976,1725,0.182159
1,Aditya Birla Sun Life,2020-21,6474,11,0.169910
2,Bandhan,2020-21,401,0,0.000000
3,Ageas Federal,2020-21,1805,50,2.770083
4,Aviva,2020-21,1055,0,0.000000
5,Bajaj Allianz,2020-21,14333,5,0.034885
6,Bharti Axa,2020-21,1893,0,0.000000
7,Canara HSBC,2020-21,1899,25,1.316482
8,Edelweiss Life,2020-21,502,2,0.398406
9,Exide Life,2020-21,5052,63,1.247031


In [70]:
df['Pending_Rate'].isna().sum()

np.int64(6)

In [71]:
np.isinf(df['Pending_Rate']).sum()

np.int64(0)

----

## 7. Long Pending Claims Share (Long_Pending_Share)

### Why are we creating this feature?

Pending_Rate tells us how many claims remain unresolved at the end of
the financial year, but it does not tell us how old those pending claims
are.

Long_Pending_Share measures the proportion of year-end pending claims
that have remained unresolved for more than one year.

This adds an ageing perspective to the claims analysis and helps identify
insurers with a potentially higher concentration of long-outstanding
claims.

### Formula

Long_Pending_Share =
(Pending Claims > 1 Year / Total Pending Claims at End) × 100

### Business Interpretation

- Lower share → most pending claims are relatively recent.
- Higher share → a larger proportion of pending claims are more than
  one year old.

This metric complements Pending_Rate:
Pending_Rate measures the **size of the pending burden**, while
Long_Pending_Share measures the **age of that burden**.

### Data Handling

If total pending claims at year-end is zero, the denominator is replaced
with `NaN` because the ageing share cannot be calculated when there are
no pending claims.

----

In [72]:
df['Long_Pending_Share'] = (
    df['pending_gt_1y'] /
    df['pending_end_count'].replace(0, np.nan)
) * 100

In [73]:
df[
    ['Insurer', 'FY',
     'pending_end_count',
     'pending_gt_1y',
     'Long_Pending_Share']
].head(10)

,Insurer,FY,pending_end_count,pending_gt_1y,Long_Pending_Share
0,LIC,2020-21,1725,0.0,0.0
1,Aditya Birla Sun Life,2020-21,11,0.0,0.0
2,Bandhan,2020-21,0,0.0,NaN
3,Ageas Federal,2020-21,50,0.0,0.0
4,Aviva,2020-21,0,0.0,NaN
5,Bajaj Allianz,2020-21,5,0.0,0.0
6,Bharti Axa,2020-21,0,0.0,NaN
7,Canara HSBC,2020-21,25,0.0,0.0
8,Edelweiss Life,2020-21,2,0.0,0.0
9,Exide Life,2020-21,63,0.0,0.0


In [74]:
df['Long_Pending_Share'].isna().sum()

np.int64(38)

In [75]:
np.isinf(df['Long_Pending_Share']).sum()

np.int64(0)

In [76]:
df['pending_gt_1y'].gt(0).sum()

np.int64(18)

In [77]:
df.loc[
    df['pending_gt_1y'] > 0,
    ['Insurer', 'FY', 'pending_end_count', 'pending_gt_1y', 'Long_Pending_Share']
].head(20)

,Insurer,FY,pending_end_count,pending_gt_1y,Long_Pending_Share
13,India First,2020-21,15,7.0,46.666667
14,Kotak Mahindra,2020-21,16,7.0,43.750000
20,SBI Life,2020-21,912,1.0,0.109649
36,ICICI Prudential,2021-22,80,3.0,3.750000
37,India First,2021-22,9,6.0,66.666667
38,Kotak Mahindra,2021-22,24,8.0,33.333333
59,ICICI Prudential,2022-23,20,8.0,40.000000
60,India First,2022-23,10,6.0,60.000000
61,Kotak Mahindra,2022-23,11,3.0,27.272727
85,ICICI Prudential,2023-24,3,2.0,66.666667


In [78]:
df['Long_Pending_Share'].max()

np.float64(100.0)

---

## 8. Claim Repudiation Rate (Repudiation_Rate)

### Why are we creating this feature?

Claim Settlement Ratio shows the proportion of claims that were paid,
but it does not separately quantify the proportion of claims that were
repudiated.

Repudiation Rate measures the proportion of total claims that were
repudiated by the insurer.

This provides an additional claims-outcome perspective and helps us
understand how frequently claims were repudiated relative to the total
claim volume.

### Formula

Repudiation_Rate =
(Repudiated Claims Count / Total Claims Count) × 100

### Business Interpretation

- Lower Repudiation_Rate → relatively fewer claims were repudiated.
- Higher Repudiation_Rate → relatively larger proportion of claims were
  repudiated.

This metric should be interpreted alongside Count_CSR, rather than in
isolation, because paid and repudiated claims represent different claim
outcomes.

### Data Handling

If `total_claims_count` is zero, it is replaced with `NaN` before division
to avoid division-by-zero errors and misleading infinite values.

----

In [79]:
df['repudiated_count'].describe()

count     122.000000
mean      441.016393
std      1645.902275
min         0.000000
25%        16.000000
50%        57.500000
75%       109.500000
max      9137.000000
Name: repudiated_count, dtype: float64

In [80]:
df['repudiated_count'].gt(0).sum()

np.int64(116)

In [81]:
df.loc[
    df['repudiated_count'] > 0,
    ['Insurer', 'FY', 'total_claims_count', 'repudiated_count']
].head(20)

,Insurer,FY,total_claims_count,repudiated_count
0,LIC,2020-21,946976,6531
1,Aditya Birla Sun Life,2020-21,6474,116
2,Bandhan,2020-21,401,3
3,Ageas Federal,2020-21,1805,38
4,Aviva,2020-21,1055,21
5,Bajaj Allianz,2020-21,14333,213
6,Bharti Axa,2020-21,1893,18
7,Canara HSBC,2020-21,1899,30
8,Edelweiss Life,2020-21,502,13
9,Exide Life,2020-21,5052,11


In [82]:
# feature

df['Repudiation_Rate'] = (
    df['repudiated_count'] /
    df['total_claims_count'].replace(0, np.nan)
) * 100

In [83]:
df[
    ['Insurer', 'FY',
     'total_claims_count',
     'repudiated_count',
     'Repudiation_Rate']
].head(10)

,Insurer,FY,total_claims_count,repudiated_count,Repudiation_Rate
0,LIC,2020-21,946976,6531,0.689669
1,Aditya Birla Sun Life,2020-21,6474,116,1.791783
2,Bandhan,2020-21,401,3,0.748130
3,Ageas Federal,2020-21,1805,38,2.105263
4,Aviva,2020-21,1055,21,1.990521
5,Bajaj Allianz,2020-21,14333,213,1.486081
6,Bharti Axa,2020-21,1893,18,0.950872
7,Canara HSBC,2020-21,1899,30,1.579779
8,Edelweiss Life,2020-21,502,13,2.589641
9,Exide Life,2020-21,5052,11,0.217736


In [84]:
df['Repudiation_Rate'].isna().sum()

np.int64(6)

In [85]:
np.isinf(df['Repudiation_Rate']).sum()

np.int64(0)

----

# Final Feature Engineering Summary

Eight business-driven features were created to strengthen the Life Claims
Intelligence analysis.

The features cover four analytical dimensions:

1. Settlement Efficiency
   - Count_CSR
   - Value_CSR
   - Count_Value_Gap

2. Claims Outcomes
   - Repudiation_Rate
   - Claim_Value_Disparity

3. Pending Claims & Ageing
   - Pending_Rate
   - Long_Pending_Share

4. Unclaimed Funds
   - Unclaimed_Ratio

The engineered features are validated for missing and infinite values
before the final dataset is exported.

Features with mathematically undefined values are retained as NaN rather
than being artificially replaced with zero.

----

In [86]:
final_features = [
    'Count_CSR',
    'Value_CSR',
    'Count_Value_Gap',
    'Unclaimed_Ratio',
    'Pending_Rate',
    'Long_Pending_Share',
    'Repudiation_Rate'
]

feature_summary = pd.DataFrame({
    'Feature': feature_cols,
    'Missing_Count': df[feature_cols].isna().sum().values,
    'Missing_%': (df[feature_cols].isna().mean() * 100).values,
    'Infinite_Count': np.isinf(df[feature_cols]).sum().values
})

feature_summary

,Feature,Missing_Count,Missing_%,Infinite_Count
0,Count_CSR,6,4.918033,0
1,Value_CSR,6,4.918033,0
2,Count_Value_Gap,6,4.918033,0
3,Claim_Value_Disparity,103,84.426230,0
4,Unclaimed_Ratio,6,4.918033,0
5,Pending_Rate,6,4.918033,0
6,Long_Pending_Share,38,31.147541,0
7,Repudiation_Rate,6,4.918033,0


---

# Final Feature Engineering Summary

Eight candidate features were evaluated during the feature engineering
process.

After assessing business relevance and data coverage, seven features were
retained in the final analytical dataset.

### Final Retained Features

#### Settlement Efficiency
- Count_CSR
- Value_CSR
- Count_Value_Gap

#### Claims Outcomes
- Repudiation_Rate

#### Pending Claims & Ageing
- Pending_Rate
- Long_Pending_Share

#### Unclaimed Funds
- Unclaimed_Ratio

### Excluded Feature

Claim_Value_Disparity was evaluated but excluded from the final dataset
because 84.4% of insurer-year observations were mathematically undefined
due to zero rejected claims.

This feature-selection process ensures that the final analytical dataset
contains metrics with sufficient data coverage and clear business
relevance.

---

### dropping claim value disparity

In [87]:
df = df.drop(columns=['Claim_Value_Disparity'])

---

# Final Csv

In [88]:
df.to_csv(
    'Life_Claims_Feature_Engineered.csv',
    index=False
)

print("Final feature-engineered dataset saved successfully.")
print("Final shape:", df.shape)

Final feature-engineered dataset saved successfully.
Final shape: (122, 30)


In [89]:
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['FY', 'Insurer', 'pending_start_count', 'pending_start_amount',
       'intimated_count', 'intimated_amount', 'total_claims_count',
       'total_claims_amount', 'paid_count', 'paid_amount', 'repudiated_count',
       'repudiated_amount', 'rejected_count', 'rejected_amount',
       'unclaimed_count', 'unclaimed_amount', 'pending_end_count',
       'pending_end_amount', 'pending_lt_3m', 'pending_3_to_6m',
       'pending_6m_to_1y', 'pending_gt_1y', 'Low_Volume_Flag', 'Count_CSR',
       'Value_CSR', 'Count_Value_Gap', 'Unclaimed_Ratio', 'Pending_Rate',
       'Long_Pending_Share', 'Repudiation_Rate'],
      dtype='str')>

----

# Thank you ! 